# SORA.Earth - Model Training & Evaluation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from xgboost import XGBClassifier
sns.set_theme(style='whitegrid')

## 1. Данные

In [ ]:
df=pd.read_csv('../data/projects.csv')
FEAT=['budget','co2_reduction','social_impact','duration_months']
X=df[FEAT]
y=df['success']
print(f'Dataset: {df.shape}, Success rate: {y.mean()*100:.1f}%')
df.head()

## 2. Масштабирование и разбиение

In [ ]:
scaler=StandardScaler()
X_sc=pd.DataFrame(scaler.fit_transform(X),columns=FEAT)
X_tr,X_te,y_tr,y_te=train_test_split(X_sc,y,test_size=0.2,random_state=42,stratify=y)
print(f'Train: {len(X_tr)}, Test: {len(X_te)}')

## 3. Модели и кросс-валидация

In [ ]:
models={'RF':RandomForestClassifier(n_estimators=200,max_depth=10,random_state=42),
  'XGB':XGBClassifier(n_estimators=200,max_depth=6,lr=0.1,use_label_encoder=False,eval_metric='logloss',random_state=42),
  'GB':GradientBoostingClassifier(n_estimators=200,max_depth=5,random_state=42),
  'LR':LogisticRegression(max_iter=1000,random_state=42)}
cv=StratifiedKFold(5,shuffle=True,random_state=42)
cv_res={}
for n,m2 in models.items():
    sc=cross_val_score(m2,X_sc,y,cv=cv,scoring='accuracy')
    cv_res[n]={'mean':sc.mean()*100,'std':sc.std()*100}
    print(f'{n:5s} {sc.mean()*100:.1f}% +/- {sc.std()*100:.2f}%')

## 4. Обучение и метрики на тесте

In [ ]:
met={}
trained={}
for n,m2 in models.items():
    m2.fit(X_tr,y_tr)
    trained[n]=m2
    yp=m2.predict(X_te)
    ypr=m2.predict_proba(X_te)[:,1]
    met[n]={'Acc':accuracy_score(y_te,yp)*100,'Prec':precision_score(y_te,yp)*100,
            'Rec':recall_score(y_te,yp)*100,'F1':f1_score(y_te,yp)*100,'AUC':roc_auc_score(y_te,ypr)*100}
pd.DataFrame(met).round(2).T

## 5. ROC-кривые

In [ ]:
fig,ax=plt.subplots(figsize=(8,6))
for n,m2 in trained.items():
    ypr=m2.predict_proba(X_te)[:,1]
    fpr,tpr,_=roc_curve(y_te,ypr)
    ax.plot(fpr,tpr,label=f'{n} (AUC={roc_auc_score(y_te,ypr):.3f})',lw=2)
ax.plot([0,1],[0,1],'k--',alpha=.3)
ax.set_xlabel('FPR');ax.set_ylabel('TPR');ax.set_title('ROC Curves')
ax.legend();ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig('../plots/model_roc.png',dpi=150)
plt.show()

## 6. Confusion Matrices

In [ ]:
fig,axes=plt.subplots(1,4,figsize=(20,4))
for ax,(n,m2) in zip(axes,trained.items()):
    cm=confusion_matrix(y_te,m2.predict(X_te))
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=ax,xticklabels=['F','S'],yticklabels=['F','S'])
    ax.set_title(f'{n} ({met[n][chr(65)+chr(99)+chr(99)]:.1f}%)')
plt.tight_layout()
plt.savefig('../plots/model_cm.png',dpi=150)
plt.show()

## 7. Feature Importance

In [ ]:
best=trained['XGB']
fi=best.feature_importances_
idx=np.argsort(fi)
plt.figure(figsize=(8,5))
plt.barh([FEAT[i] for i in idx],fi[idx],color=['#00e5a0','#00c2ff','#8b5cf6','#eab308'])
plt.xlabel('Importance');plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.savefig('../plots/model_fi.png',dpi=150)
plt.show()

## 7. Feature Importance

In [ ]:
best=trained['XGB']
fi=best.feature_importances_
idx=np.argsort(fi)
plt.figure(figsize=(8,5))
plt.barh([FEAT[i] for i in idx],fi[idx],color=['#00e5a0','#00c2ff','#8b5cf6','#eab308'])
plt.xlabel('Importance');plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.savefig('../plots/model_fi.png',dpi=150)
plt.show()

## 8. Сравнение моделей

In [ ]:
mnames=list(met.keys())
metrics=['Acc','Prec','Rec','F1','AUC']
x=np.arange(len(metrics))
w=0.2
fig,ax=plt.subplots(figsize=(10,5))
for i,n in enumerate(mnames):
    vals=[met[n][m3] for m3 in metrics]
    ax.bar(x+i*w,vals,w,label=n)
ax.set_xticks(x+w*1.5);ax.set_xticklabels(metrics)
ax.set_ylim(50,100);ax.legend();ax.grid(axis='y',alpha=.3)
ax.set_title('Model Comparison')
plt.tight_layout()
plt.savefig('../plots/model_comp.png',dpi=150)
plt.show()

## 9. Выводы

In [ ]:
best_n=max(cv_res,key=lambda k:cv_res[k]['mean'])
print(f'Best model: {best_n}')
print(f'CV accuracy: {cv_res[best_n][chr(109)+chr(101)+chr(97)+chr(110)]:.1f}%')
print(f'Test metrics:')
for k,v in met[best_n].items():
    print(f'  {k}: {v:.2f}%')